# The regime classifier fitted on the training participants

Notebook 06's classifier (E34) was fitted on the four validation participants only, because only they and the test
participants had every window exported. The 14 training participants are now encoded too
(`ml/nature/awareness/encode_split.py --split train`), so the same classifier can be fitted on 3.5x as many people.

**The catch.** Both encoders were trained on these 14 people: the LSTM-VAE on their ID-safe windows, the TCN-VAE on their
ID-safe and OOD-safe windows. Their features are in-sample there, so familiar windows of training people may look *more*
familiar than a new wearer's. A classifier fitted on them could learn boundaries that are too tight. Section 2 measures
that shift before anything is fitted, and the novelty scores' own fit windows (in-sample by construction: LOF finds each one
among its own neighbours) are left out of fitting.

| | |
|---|---|
| labels | `controller_class`: the provisional regimes of E18, set from where the JME paper (Molinaro, Scherpereel et al., *Nature* 2024) reported its estimator working well or badly per task |
| classifier | notebook 06's choice, not re-tuned: `HistGradientBoostingClassifier`, balanced class weights, 300 iterations, random state = seed |
| features | the same 42: TCN-VAE posterior means (ID + OOD-safe, 3 s + spectral, β 128), LSTM-VAE LOF / Mahalanobis / posterior variance (ID, 3 s raw, β 1), 7 rhythm features |
| fits compared | **val** (4 people, E34), **train** (14), **train + val** (18) |
| decision | probabilities → 0.5 s causal median by recording → argmax regime (E34); for the train fit also *u* with two cutoffs chosen on the validation participants (E35) |
| test | the 4 test participants, scored once; 3 seeds (encoder seed = classifier seed) |
| held-out dangerous tasks | each of the 4 dangerous tasks left out of fitting in turn; where its test windows go |

The protocol above was fixed before any classifier was fitted on the training participants.

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml' / 'nature' / 'nature_data.py').exists())
for folder in ('ml/common', 'ml/nature', 'notebooks/nature/model_exploration/onboard'):
    if str(REPO / folder) not in sys.path:
        sys.path.insert(0, str(REPO / folder))
import _novelty_scores as ns
import _regime_classifier as rc

ROOT = REPO / 'notebooks' / 'nature' / 'model_exploration'
DATA = REPO / 'data' / 'processed_nature' / 'ae'
CACHE = REPO / 'data' / 'cache' / 'nature' / 'novelty'
FIG_DIR, RESULTS = Path.cwd() / 'figures', Path.cwd() / 'results'
SEEDS = (0, 1, 2)
SPLITS = ('train', 'val', 'test')
ENCODER = ROOT / 'runs_train_class' / 'id_ood_safe' / 'vae' / 'tcn_vae_raw_3s_spectral_b128_fixed_s{seed}'   # notebook 06's
DETECTOR = ROOT / 'runs' / 'vae' / 'lstm_vae_raw_3s_b1_fixed_s{seed}'
NOVELTY = ['LOF, raw latent', 'Mahalanobis, latent', 'posterior variance']
FITS = {'val': ('val',), 'train': ('train',), 'train + val': ('train', 'val')}
REGIME_C = dict(zip(rc.REGIMES, ['#287da8', '#dd791a', '#a32d2d']))     # notebook 01's controller-class colours
pd.set_option('display.width', 220)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 9})

meta = json.loads((DATA / 'meta.json').read_text())
tables = {k: pd.read_parquet(DATA / meta['inputs'][k]['table']) for k in ('raw_3s', 'raw_3s_spectral')}
rows = {s: rc.get_split_rows(tables['raw_3s'], s) for s in SPLITS}
keys = {s: r[['rec', 'end']] for s, r in rows.items()}
y = {s: r['controller_class'].astype(str).to_numpy() for s, r in rows.items()}
for s in SPLITS:
    print(f"{s}: {len(y[s]):,} windows, {rows[s]['subject'].nunique()} participants; "
          + ', '.join(f'{c} {np.mean(y[s] == c):.0%}' for c in rc.REGIMES))

## 1. Features for all three splits

Validation and test come from the existing exports (`rc.get_latents`, `rc.get_novelty`, as in notebooks 06 and 12); the
training participants from `latents_train.npz`. The check: scoring the validation export through the new training-split
path gives the same novelty scores as notebook 06's path.

In [ ]:
t0 = time.time()
x, in_sample = {}, {}
for seed in SEEDS:
    enc, det = Path(str(ENCODER).format(seed = seed)), Path(str(DETECTOR).format(seed = seed))
    vt = {s: keys[s] for s in ('val', 'test')}
    lat, nov = rc.get_latents(enc, tables['raw_3s_spectral'], vt), rc.get_novelty(det, tables['raw_3s'], vt, NOVELTY, CACHE)
    lat['train'] = rc.get_split_latents(enc, tables['raw_3s_spectral'], keys['train'], 'train')
    nov['train'] = rc.get_split_novelty(det, tables['raw_3s'], keys['train'], NOVELTY, 'train', CACHE)
    np.testing.assert_allclose(rc.get_split_novelty(det, tables['raw_3s'], keys['val'], NOVELTY, 'val', CACHE), nov['val'], rtol = 1e-6)
    x[seed] = {s: np.concatenate([lat[s], nov[s], rows[s][rc.RHYTHM].to_numpy(dtype = np.float64)], axis = 1) for s in SPLITS}
    fit_windows = rc.get_score_fit_windows(det, tables['raw_3s'])
    in_sample[seed] = keys['train'].merge(fit_windows.assign(hit = True), on = ['rec', 'end'], how = 'left')['hit'].notna().to_numpy()
    assert all(np.isfinite(v).all() for v in x[seed].values())
print({s: x[0][s].shape for s in SPLITS}, f'novelty fit windows left out of the train fit: {in_sample[0].sum():,}',
      f'(all ID-safe: {set(y["train"][in_sample[0]])}); {time.time() - t0:.0f} s')

## 2. Are the training participants' features in-sample?

For each novelty score, its median per regime and split, and the share of each split's ID-safe windows above the
99th percentile of the validation participants' ID-safe windows (1% by construction on validation; a new wearer, the test
participants, shows how much that drifts anyway). A training-participant share well below the test share means the
encoder knows those people. For the TCN-VAE latent, the same comparison on its Mahalanobis distance to the validation
OOD-safe + ID-safe windows (the classes it was trained on).

In [ ]:
def get_shift(values: dict[str, np.ndarray], familiar: str | tuple[str, ...]) -> dict[str, float]:
    # share (%) of each split's familiar windows above the validation familiar windows' 99th percentile
    fam = {s: np.isin(y[s], familiar) for s in SPLITS}
    cut = np.percentile(values['val'][fam['val']], 99)
    return {s: 100 * float(np.mean(values[s][fam[s]] > cut)) for s in SPLITS}


shift_rows, median_rows = [], []
for seed in SEEDS:
    for j, name in enumerate(NOVELTY):
        col = {s: x[seed][s][:, 32 + j] for s in SPLITS}
        sign = 1 if np.median(col['val'][y['val'] != ns.ID]) > np.median(col['val'][y['val'] == ns.ID]) else -1   # higher = less familiar
        shift_rows.append({'seed': seed, 'feature': f'LSTM-VAE {name}', **get_shift({s: sign * v for s, v in col.items()}, ns.ID)})
        for s in SPLITS:
            median_rows.append({'seed': seed, 'feature': name, 'split': s, **{r: float(np.median(col[s][y[s] == r])) for r in rc.REGIMES}})
    lat = {s: x[seed][s][:, :32] for s in SPLITS}
    trained = np.isin(y['val'], (ns.ID, ns.SAFE))
    maha = dict(zip(SPLITS, ns.score_mahalanobis(lat['val'][trained], [lat[s] for s in SPLITS])))
    shift_rows.append({'seed': seed, 'feature': 'TCN-VAE latent, Mahalanobis', **get_shift(maha, (ns.ID, ns.SAFE))})
shift = pd.DataFrame(shift_rows).groupby('feature')[list(SPLITS)].agg(['mean', 'std'])
medians = pd.DataFrame(median_rows).groupby(['feature', 'split'], sort = False)[list(rc.REGIMES)].mean()
print('familiar windows above the validation familiar 99th percentile (%), mean ± SD over seeds:')
display(pd.DataFrame({s: shift[(s, 'mean')].map('{:.2f}'.format) + ' ± ' + shift[(s, 'std')].map('{:.2f}'.format) for s in SPLITS}))
display(medians.round(4))
pd.DataFrame(shift_rows).to_csv(RESULTS / '13_in_sample_shift.csv', index = False)

## 3. Fit on validation, training, and training + validation; score the test participants once

In [ ]:
def fit(seed: int, splits: tuple[str, ...], drop_task: str | None = None) -> HistGradientBoostingClassifier:
    # notebook 06's classifier on some splits' windows, without the novelty scores' own fit windows (in-sample)
    xs, ys = [], []
    for s in splits:
        keep = ~in_sample[seed] if s == 'train' else np.ones(len(y[s]), bool)
        if drop_task is not None:
            keep &= rows[s]['task'].astype(str).to_numpy() != drop_task
        xs.append(x[seed][s][keep])
        ys.append(y[s][keep])
    model = HistGradientBoostingClassifier(class_weight = 'balanced', max_iter = 300, random_state = seed)
    return model.fit(np.concatenate(xs), np.concatenate(ys))


def get_proba(model, xs: np.ndarray, labels: pd.DataFrame) -> np.ndarray:
    # class probabilities in REGIMES order after the 0.5 s causal median by recording. (N, F) -> (N, 3)
    p = pd.DataFrame(model.predict_proba(xs), columns = model.classes_).reindex(columns = list(rc.REGIMES), fill_value = 0).to_numpy()
    return rc.get_smoothed(p, labels)


t0 = time.time()
models, score_rows, person_rows, proba = {}, [], [], {}
for seed in SEEDS:
    for name, splits in FITS.items():
        models[seed, name] = fit(seed, splits)
        proba[seed, name] = get_proba(models[seed, name], x[seed]['test'], rows['test'])
        pred = np.array(rc.REGIMES)[proba[seed, name].argmax(axis = 1)]
        n_fit = sum(len(y[s]) - (in_sample[seed].sum() if s == 'train' else 0) for s in splits)
        score_rows.append({'fit': name, 'seed': seed, 'fit windows': n_fit, **rc.get_regime_metrics(y['test'], pred)})
        for person in sorted(rows['test']['subject'].astype(str).unique()):
            on = (rows['test']['subject'].astype(str) == person).to_numpy()
            person_rows.append({'fit': name, 'seed': seed, 'participant': person, **rc.get_regime_metrics(y['test'][on], pred[on])})
        print(f'seed {seed}, fit {name}: {time.time() - t0:.0f} s')
scores, persons = pd.DataFrame(score_rows), pd.DataFrame(person_rows)
scores.to_csv(RESULTS / '13_regime_scores.csv', index = False)
persons.to_csv(RESULTS / '13_regime_per_participant.csv', index = False)
ref = pd.read_csv(RESULTS / '06_regime_scores.csv')
ref = ref[(ref['model'] == 'boosted trees') & (ref['features'] == 'latent + novelty + rhythm')].set_index('seed')['test macro F1']
np.testing.assert_allclose(scores[scores['fit'] == 'val'].set_index('seed')['macro F1'].to_numpy(), ref.loc[list(SEEDS)].to_numpy(), atol = 1e-9)


def show(frame: pd.DataFrame, by: str | list[str]) -> pd.DataFrame:
    # 'mean ± sd' over seeds
    s = frame.drop(columns = 'seed').groupby(by, sort = False).agg(['mean', 'std'])
    cols = dict.fromkeys(c for c, _ in s.columns)
    return pd.DataFrame({c: s[(c, 'mean')].map('{:.3f}'.format) + ' ± ' + s[(c, 'std')].map('{:.3f}'.format) for c in cols})


print('val fit reproduces notebook 06 exactly')
display(show(scores, 'fit'))
display(persons.groupby(['participant', 'fit'], sort = False)['macro F1'].mean().unstack('fit')[list(FITS)].round(3))

Where each true regime's test windows go (mean over seeds), and per task.

In [ ]:
for name in FITS:
    conf = np.mean([rc.get_confusion(y['test'], np.array(rc.REGIMES)[proba[seed, name].argmax(axis = 1)]).to_numpy() for seed in SEEDS], axis = 0)
    print(name)
    display(pd.DataFrame(conf, index = pd.Index(rc.REGIMES, name = 'true'), columns = pd.Index(rc.REGIMES, name = 'given')).round(1))
task_rows = []
for seed in SEEDS:
    for name in FITS:
        pred = np.array(rc.REGIMES)[proba[seed, name].argmax(axis = 1)]
        for task, on in rows['test'].groupby(rows['test']['task'].astype(str)).groups.items():
            task_rows.append({'fit': name, 'seed': seed, 'task': task, 'true regime': y['test'][on[0]],
                              **{f'given {r} (%)': 100 * np.mean(pred[on] == r) for r in rc.REGIMES}})
per_task = pd.DataFrame(task_rows)
per_task.to_csv(RESULTS / '13_regime_per_task.csv', index = False)
cols = [f'given {r} (%)' for r in rc.REGIMES]
display(per_task.groupby(['true regime', 'task', 'fit'], sort = False)[cols].mean().unstack('fit').round(1))

## 4. The uncertainty score *u* with cutoffs chosen on the validation participants

The training fit leaves the validation participants unused, so E35's two cutoffs on
*u* = 0.5·p<sub>OOD-safe</sub> + p<sub>dangerous</sub> can be chosen on them (macro F1, 0.005 grid) and then applied to the
test participants once.

In [ ]:
u_rows = []
for seed in SEEDS:
    u_val = rc.get_uncertainty(get_proba(models[seed, 'train'], x[seed]['val'], rows['val']))
    cutoffs = rc.get_cutoffs(u_val, y['val'])
    u_test = rc.get_uncertainty(proba[seed, 'train'])
    u_rows.append({'seed': seed, 'cutoff 1': cutoffs[0], 'cutoff 2': cutoffs[1],
                   **rc.get_regime_metrics(y['test'], rc.get_banded(u_test, cutoffs)),
                   'dangerous in the estimator band (%)': 100 * np.mean(rc.get_banded(u_test[y['test'] == ns.DANGEROUS], cutoffs) == ns.SAFE)})
u_scores = pd.DataFrame(u_rows)
u_scores.to_csv(RESULTS / '13_uncertainty_scores.csv', index = False)
display(show(u_scores.assign(fit = 'train, u cutoffs on val'), 'fit'))

## 5. Dangerous tasks never seen in fitting

Each dangerous task is removed from every fitting split in turn; its test windows are then classified. E34 (val fit): under
0.5% to the spline, 30 to 38% switched off.

In [ ]:
t0 = time.time()
DANGEROUS_TASKS = sorted(rows['test'].loc[y['test'] == ns.DANGEROUS, 'task'].astype(str).unique())
held_rows = []
for seed in SEEDS:
    for name in ('val', 'train'):
        for task in DANGEROUS_TASKS:
            on = (rows['test']['task'].astype(str) == task).to_numpy()
            model = fit(seed, FITS[name], drop_task = task)
            pred = np.array(rc.REGIMES)[get_proba(model, x[seed]['test'][on], rows['test'][on].reset_index(drop = True)).argmax(axis = 1)]
            held_rows.append({'fit': name, 'seed': seed, 'task': task, **{f'given {r} (%)': 100 * np.mean(pred == r) for r in rc.REGIMES}})
        print(f'seed {seed}, fit {name}: {time.time() - t0:.0f} s')
held = pd.DataFrame(held_rows)
held.to_csv(RESULTS / '13_held_out_dangerous.csv', index = False)
display(held.groupby(['task', 'fit'])[cols].mean().unstack('fit').round(1))
display(held.groupby('fit')[cols].mean().round(1))

## 6. Figure

In [ ]:
fig, (ax_s, ax_p, ax_h) = plt.subplots(1, 3, figsize = (15, 4.2), gridspec_kw = {'width_ratios': [1.4, 1, 1]})
metrics = ['macro F1', 'recall, ID-safe', 'recall, OOD-safe', 'recall, OOD-dangerous']
fit_c = {'val': '#9a9a9a', 'train': '#4a3aa7', 'train + val': '#1baf7a'}
width = .26
for i, name in enumerate(FITS):
    s = scores[scores['fit'] == name]
    ax_s.bar(np.arange(len(metrics)) + (i - 1) * width, s[metrics].mean(), width, yerr = s[metrics].std(), color = fit_c[name],
             label = f'{name} ({rows[FITS[name][0]]["subject"].nunique() + (4 if name == "train + val" else 0)} people)', capsize = 2)
ax_s.set_xticks(range(len(metrics)), [m.replace(', ', '
') for m in metrics])
ax_s.set_ylim(0.5, 1)
ax_s.set_title('Test participants: fitted on', loc = 'left', fontweight = 'bold')
ax_s.legend(frameon = False, loc = 'lower left')
pp = persons.groupby(['participant', 'fit'])['macro F1'].mean().unstack('fit')[list(FITS)]
for person, r in pp.iterrows():
    ax_p.plot(range(len(FITS)), r.to_numpy(), marker = 'o', color = '#555', lw = 1)
    ax_p.annotate(person, (len(FITS) - 1, r.iloc[-1]), xytext = (4, 0), textcoords = 'offset points', va = 'center', fontsize = 8)
ax_p.set_xticks(range(len(FITS)), list(FITS))
ax_p.set_title('Macro F1 per test participant', loc = 'left', fontweight = 'bold')
hm = held.groupby('fit')[cols].mean().loc[['val', 'train']]
bottom = np.zeros(len(hm))
for r, c in zip(rc.REGIMES, cols):
    ax_h.bar(range(len(hm)), hm[c], bottom = bottom, color = REGIME_C[r], label = f'given {r}', width = .6)
    bottom += hm[c].to_numpy()
ax_h.set_xticks(range(len(hm)), [f'fit {n}' for n in hm.index])
ax_h.set_ylabel('windows of the unseen dangerous task (%)')
ax_h.set_title('Dangerous task left out of fitting', loc = 'left', fontweight = 'bold')
ax_h.legend(frameon = False, fontsize = 8, loc = 'upper center', bbox_to_anchor = (.5, -.1), ncol = 3)
fig.tight_layout()
fig.savefig(FIG_DIR / '13_classifier_training_participants.png', dpi = 200, bbox_inches = 'tight')
plt.show()

## Findings

Test participants, mean ± SD over 3 seeds (write-up: `docs/experiments/E39_nature_classifier_training_participants.md`).

- **The training participants' features are only mildly in-sample.** Of their familiar windows, 0.5% exceed the
  validation familiar 99th percentile on LOF, 1.9% on Mahalanobis and 3.0% on posterior variance, against 0.9%, 2.2% and
  4.6% for the test participants. So training people look a little more familiar than new wearers, but the gap is smaller
  than the validation-to-test drift (1% → 2–5%) between two sets of new people. The TCN-VAE latent shows no in-sample
  advantage (1.8% against 1.5%). Per-regime medians are almost identical across splits. Cross-fitting the encoders is not
  needed at this scale.
- **Macro F1 does not move** (val 0.868 ± 0.005, train 0.860 ± 0.014, train + val 0.869 ± 0.013), but **the trade-off
  does**. Fitted on 14 people, the argmax decision switches off 88% of dangerous windows (79% for the val fit) and 5.1% of
  safe ones (2.8%). Still no dangerous window reaches the spline (train + val: 0.02%). Stairs go to the spline 78–79% of
  the time (59% for the val fit). Squats, lifting, poses and twister account for the extra false shutdowns.
- **With *u* and cutoffs chosen on the now-unused validation participants** (cutoffs 0.21 and 0.85), the training fit
  gives the best decision so far: macro F1 **0.875 ± 0.006**, dangerous recall 0.82, 2.7% of safe windows switched off,
  no dangerous window on the spline, and 18% of dangerous windows in the estimator band (E35: 0.868, 2.1%, 24%).
- **Unseen dangerous tasks are switched off more often:** 41% of their windows against 26% for the val fit (ball toss 61%
  against 24%, sit-to-stand 62% against 47%, tug of war 38% against 31%). Step-ups stay invisible (3% against 1%). Almost
  none go to the spline (at most 0.2%, ball toss).
- **Per test participant** the F1 spread is unchanged (0.85–0.87), so more people did not make the classifier
  more uniform across wearers.